# 🔍 Notebook 7 — Community Search Engine
## Fake Internship & Job Scam Detection System

---

### 📌 Project Overview

| Detail | Info |
|--------|------|
| **Project** | Fake Internship & Job Scam Detection System |
| **Notebook** | 07 — Community Search Engine |
| **Dependencies** | Notebook 5 (Detection Engine), Notebook 6 (URL Analysis Engine) |
| **Purpose** | Keyword + Fuzzy Matching Search across Community Data |
| **Search Targets** | Community Posts · User Comments · Scam Reports |

---

### 🎯 Objective

When a user searches **"ABC Consulting"**, the engine returns all related:
- 📝 Community Posts
- 💬 User Comments
- 🚨 Scam Reports

...even when names vary slightly:

```
ABC Consulting  →  ABC Consultancy
                →  ABC Consulting Pvt Ltd
                →  ABC Career Consulting
```

---

### 📂 Notebook Flow

```
STEP 01 → Import Libraries
STEP 02 → Load Community Data
STEP 03 → Data Cleaning & Normalization
STEP 04 → Build Search Corpus
STEP 05 → Exact Match Search
STEP 06 → Fuzzy Match Search
STEP 07 → Company Search Engine
STEP 08 → Popularity Ranking
STEP 09 → Community Intelligence Summary
STEP 10 → API-Style Response
STEP 11 → Test Cases
STEP 12 → Export Search Index
```

---
## STEP 01 — Import Libraries

All required libraries for data processing, text matching, and search functionality.

In [16]:
# ============================================================
#  STEP 01 — IMPORT LIBRARIES
#  Community Search Engine | Notebook 7
# ============================================================

# Core data processing
import pandas as pd
import numpy as np

# Text processing
import re
import json

# Fuzzy string matching — high-performance Levenshtein-based matching
from rapidfuzz import fuzz, process

# Standard utilities
import warnings
import os
from datetime import datetime

# Display settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 120)

print("All libraries imported successfully.")
print(f"   pandas     : {pd.__version__}")
print(f"   numpy      : {np.__version__}")
print(f"   rapidfuzz  : (fuzzy matching ready)")
print(f"   re / json  : (stdlib — always available)")

All libraries imported successfully.
   pandas     : 2.2.2
   numpy      : 1.26.4
   rapidfuzz  : (fuzzy matching ready)
   re / json  : (stdlib — always available)


---
## STEP 02 — Load Community Data

Load the three primary data sources:
- `community_posts.csv`
- `community_comments.csv`
- `scam_reports.csv`

If files are not present, realistic synthetic data is generated automatically.

In [17]:
# ============================================================
#  STEP 02 — LOAD COMMUNITY DATA
# ============================================================

DATA_DIR = "../data/"
os.makedirs(DATA_DIR, exist_ok=True)

# ----------------------------------------------------------
#  Synthetic data generator
#  Used when CSV files are not yet present in ../data/
# ----------------------------------------------------------

def generate_synthetic_posts():
    """Generate realistic community posts about job/internship scams."""
    records = [
        {"post_id": "P001", "company_name": "ABC Consulting",         "title": "Beware of ABC Consulting fake internship offer",         "content": "They asked me to pay a registration fee of Rs 2000 to secure an internship. Red flag!",         "author": "user_rahul",  "date": "2024-01-15", "likes": 45, "number_of_comments": 12, "tags": "scam,internship,registration fee"},
        {"post_id": "P002", "company_name": "ABC Consultancy",        "title": "ABC Consultancy demanded money before joining",            "content": "ABC Consultancy Pvt Ltd sent me an offer letter but asked security deposit first.",         "author": "user_priya",  "date": "2024-01-18", "likes": 38, "number_of_comments": 9,  "tags": "scam,money,security deposit"},
        {"post_id": "P003", "company_name": "TCS",                    "title": "Fake TCS offer letter circulating on WhatsApp",           "content": "Someone is sending TCS offer letters via WhatsApp asking for Rs 5000 processing fee.",  "author": "user_amit",  "date": "2024-01-20", "likes": 102,"number_of_comments": 34, "tags": "TCS,whatsapp,fake offer"},
        {"post_id": "P004", "company_name": "TCS",                    "title": "TCS does not charge any fees — official clarification",   "content": "TCS official website clearly states they never ask candidates to pay any fee.",          "author": "user_neha",  "date": "2024-01-22", "likes": 87, "number_of_comments": 21, "tags": "TCS,official,clarification"},
        {"post_id": "P005", "company_name": "Infosys",                "title": "Fake Infosys internship on LinkedIn",                    "content": "Received a message from Infosys HR on LinkedIn asking to pay Rs 3000 for training kit.",  "author": "user_kiran", "date": "2024-02-01", "likes": 67, "number_of_comments": 18, "tags": "Infosys,LinkedIn,training fee"},
        {"post_id": "P006", "company_name": "Infosys BPM",            "title": "Infosys BPM scam — registration fee demanded",           "content": "Infosys BPM HR contacted me via Gmail asking registration fee before onboarding.",         "author": "user_maya",  "date": "2024-02-03", "likes": 55, "number_of_comments": 14, "tags": "Infosys,BPM,registration fee"},
        {"post_id": "P007", "company_name": "ABC Career Consulting",  "title": "ABC Career Consulting is a scam company",               "content": "ABC Career Consulting promised placement but charged Rs 10000 upfront. No job given.",     "author": "user_suresh","date": "2024-02-10", "likes": 74, "number_of_comments": 23, "tags": "scam,placement,money"},
        {"post_id": "P008", "company_name": "WhatsApp Jobs",          "title": "Jobs offered via WhatsApp groups are mostly fake",       "content": "Never pay anyone who contacts you through WhatsApp for job or internship offers.",       "author": "user_anita", "date": "2024-02-12", "likes": 120,"number_of_comments": 45, "tags": "whatsapp,fake,awareness"},
        {"post_id": "P009", "company_name": "Global Placement Hub",   "title": "Global Placement Hub charged and disappeared",          "content": "Paid Rs 8000 registration fee to Global Placement Hub. They stopped responding afterwards.","author": "user_raj",   "date": "2024-02-15", "likes": 33, "number_of_comments": 7,  "tags": "scam,placement,registration fee"},
        {"post_id": "P010", "company_name": "ABC Consulting Pvt Ltd", "title": "ABC Consulting Pvt Ltd — another victim here",           "content": "They rebranded from ABC Consulting. Same scam pattern. Asking for registration fees.",    "author": "user_deepa", "date": "2024-02-18", "likes": 41, "number_of_comments": 11, "tags": "scam,rebrand,registration fee"},
        {"post_id": "P011", "company_name": "TCS iON",                "title": "Fake TCS iON internship offer via email",               "content": "Received email from tcs-ion-hr@gmail.com asking for fee. TCS uses @tcs.com only.",        "author": "user_vikram","date": "2024-02-20", "likes": 89, "number_of_comments": 29, "tags": "TCS,iON,fake email"},
        {"post_id": "P012", "company_name": "Infosys",                "title": "How to verify genuine Infosys offer letter",            "content": "Infosys provides verification link in offer letters. Always cross-check on official portal.","author": "user_leena", "date": "2024-02-22", "likes": 95, "number_of_comments": 31, "tags": "Infosys,verification,guide"},
    ]
    return pd.DataFrame(records)


def generate_synthetic_comments():
    """Generate realistic community comments referencing post IDs."""
    records = [
        {"comment_id": "C001", "post_id": "P001", "company_name": "ABC Consulting",        "comment": "I also got the same message from ABC Consulting. Definitely a scam!",               "author": "user_arun",   "date": "2024-01-16", "upvotes": 15},
        {"comment_id": "C002", "post_id": "P001", "company_name": "ABC Consulting",        "comment": "Report them to cybercrime.gov.in. ABC Consulting is listed there already.",         "author": "user_sara",   "date": "2024-01-16", "upvotes": 22},
        {"comment_id": "C003", "post_id": "P002", "company_name": "ABC Consultancy",       "comment": "Same experience. ABC Consultancy asked for security deposit via Google Pay.",       "author": "user_mohit",  "date": "2024-01-19", "upvotes": 18},
        {"comment_id": "C004", "post_id": "P003", "company_name": "TCS",                   "comment": "TCS official HR never contacts via personal WhatsApp. Always through official mail.", "author": "user_geeta",  "date": "2024-01-21", "upvotes": 35},
        {"comment_id": "C005", "post_id": "P003", "company_name": "TCS",                   "comment": "The WhatsApp TCS offer letter had spelling mistakes. Classic fake.",                "author": "user_ravi",   "date": "2024-01-21", "upvotes": 28},
        {"comment_id": "C006", "post_id": "P005", "company_name": "Infosys",               "comment": "Infosys never asks for training fee. Their official careers page confirms this.",   "author": "user_pooja",  "date": "2024-02-02", "upvotes": 41},
        {"comment_id": "C007", "post_id": "P005", "company_name": "Infosys",               "comment": "I reported this fake Infosys recruiter to LinkedIn. Profile was removed.",          "author": "user_kamal",  "date": "2024-02-02", "upvotes": 19},
        {"comment_id": "C008", "post_id": "P007", "company_name": "ABC Career Consulting", "comment": "ABC Career Consulting is same group as ABC Consulting. Different names, same scam.","author": "user_tanya",  "date": "2024-02-11", "upvotes": 33},
        {"comment_id": "C009", "post_id": "P008", "company_name": "WhatsApp Jobs",         "comment": "Never trust job offers on WhatsApp. Always verify from official company website.",  "author": "user_arjun",  "date": "2024-02-13", "upvotes": 55},
        {"comment_id": "C010", "post_id": "P010", "company_name": "ABC Consulting Pvt Ltd","comment": "They changed name to ABC Consulting Pvt Ltd but same phone number. Scammers.",      "author": "user_nisha",  "date": "2024-02-19", "upvotes": 27},
        {"comment_id": "C011", "post_id": "P011", "company_name": "TCS iON",               "comment": "TCS official domains end in @tcs.com only. Gmail IDs are always fake.",            "author": "user_rohit",  "date": "2024-02-21", "upvotes": 44},
        {"comment_id": "C012", "post_id": "P012", "company_name": "Infosys",               "comment": "Great tip! I verified my Infosys offer through their InfyTQ portal. Genuine.",      "author": "user_sonia",  "date": "2024-02-23", "upvotes": 36},
        {"comment_id": "C013", "post_id": "P001", "company_name": "ABC Consulting",        "comment": "Registration fee is the #1 red flag. Any company asking this is a scam.",          "author": "user_dev",    "date": "2024-01-17", "upvotes": 48},
        {"comment_id": "C014", "post_id": "P009", "company_name": "Global Placement Hub",  "comment": "Global Placement Hub also has fake LinkedIn profiles. Check their connections.",   "author": "user_prem",   "date": "2024-02-16", "upvotes": 14},
    ]
    return pd.DataFrame(records)


def generate_synthetic_reports():
    """Generate realistic scam reports filed by community members."""
    records = [
        {"report_id": "R001", "company_name": "ABC Consulting",        "report_type": "Fake Internship",     "description": "ABC Consulting collected Rs 2000 as registration fee for internship. Company untraceable after payment.", "reported_by": "user_rahul",  "date": "2024-01-15", "severity": "High",   "number_of_reports": 8,  "verified": True},
        {"report_id": "R002", "company_name": "ABC Consultancy",       "report_type": "Security Deposit",   "description": "ABC Consultancy demanded security deposit of Rs 5000 before issuing joining letter.",                  "reported_by": "user_priya",  "date": "2024-01-18", "severity": "High",   "number_of_reports": 5,  "verified": True},
        {"report_id": "R003", "company_name": "TCS",                   "report_type": "Impersonation",      "description": "Scammers impersonating TCS HR via WhatsApp and Gmail asking processing fees.",                         "reported_by": "user_amit",  "date": "2024-01-20", "severity": "Critical","number_of_reports": 24, "verified": True},
        {"report_id": "R004", "company_name": "Infosys",               "report_type": "Fake Offer Letter",  "description": "Forged Infosys offer letters shared on Telegram and WhatsApp with fee demand.",                       "reported_by": "user_kiran",  "date": "2024-02-01", "severity": "Critical","number_of_reports": 19, "verified": True},
        {"report_id": "R005", "company_name": "ABC Career Consulting", "report_type": "Placement Scam",     "description": "Charged Rs 10000 for guaranteed placement. No placement provided. Phones switched off.",              "reported_by": "user_suresh", "date": "2024-02-10", "severity": "High",   "number_of_reports": 11, "verified": True},
        {"report_id": "R006", "company_name": "Global Placement Hub",  "report_type": "Registration Scam", "description": "Collected fees from 50+ students promising MNC placements. Office address was fake.",                 "reported_by": "user_raj",   "date": "2024-02-15", "severity": "Critical","number_of_reports": 31, "verified": True},
        {"report_id": "R007", "company_name": "ABC Consulting Pvt Ltd","report_type": "Fake Internship",    "description": "Same operators as ABC Consulting. Rebranded after complaints. Same scam tactics.",                    "reported_by": "user_deepa",  "date": "2024-02-18", "severity": "High",   "number_of_reports": 6,  "verified": False},
        {"report_id": "R008", "company_name": "TCS iON",               "report_type": "Impersonation",      "description": "Using tcs-ion-hr@gmail.com to send fake offer letters and collect document fees.",                    "reported_by": "user_vikram", "date": "2024-02-20", "severity": "High",   "number_of_reports": 13, "verified": True},
    ]
    return pd.DataFrame(records)


# ----------------------------------------------------------
#  Load or generate data
# ----------------------------------------------------------

def load_dataset(filename, generator_fn, label):
    """Load CSV if it exists, else generate synthetic data and save it."""
    filepath = os.path.join(DATA_DIR, filename)
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        print(f"   Loaded   {label:<30} → {len(df):>4} records  [{filepath}]")
    else:
        df = generator_fn()
        df.to_csv(filepath, index=False)
        print(f"   Generated {label:<30} → {len(df):>4} records  [saved to {filepath}]")
    return df


print("\n Loading Community Data...")
print("-" * 70)

df_posts    = load_dataset("community_posts.csv",    generate_synthetic_posts,    "community_posts.csv")
df_comments = load_dataset("community_comments.csv", generate_synthetic_comments, "community_comments.csv")
df_reports  = load_dataset("scam_reports.csv",       generate_synthetic_reports,  "scam_reports.csv")

print("-" * 70)
print(f"\n Data Loading Complete.")
print(f"   Total Posts    : {len(df_posts)}")
print(f"   Total Comments : {len(df_comments)}")
print(f"   Total Reports  : {len(df_reports)}")


 Loading Community Data...
----------------------------------------------------------------------
   Loaded   community_posts.csv            →   12 records  [../data/community_posts.csv]
   Loaded   community_comments.csv         →   14 records  [../data/community_comments.csv]
   Loaded   scam_reports.csv               →    8 records  [../data/scam_reports.csv]
----------------------------------------------------------------------

 Data Loading Complete.
   Total Posts    : 12
   Total Comments : 14
   Total Reports  : 8


In [18]:
# ----------------------------------------------------------
#  Quick Preview of Each Dataset
# ----------------------------------------------------------

print("\n Community Posts — Preview")
print(df_posts[["post_id","company_name","title","likes","number_of_comments"]].to_string(index=False))

print("\n Community Comments — Preview")
print(df_comments[["comment_id","post_id","company_name","comment","upvotes"]].head(6).to_string(index=False))

print("\n Scam Reports — Preview")
print(df_reports[["report_id","company_name","report_type","severity","number_of_reports","verified"]].to_string(index=False))


 Community Posts — Preview
post_id           company_name                                                 title  likes  number_of_comments
   P001         ABC Consulting        Beware of ABC Consulting fake internship offer     45                  12
   P002        ABC Consultancy         ABC Consultancy demanded money before joining     38                   9
   P003                    TCS         Fake TCS offer letter circulating on WhatsApp    102                  34
   P004                    TCS TCS does not charge any fees — official clarification     87                  21
   P005                Infosys                   Fake Infosys internship on LinkedIn     67                  18
   P006            Infosys BPM          Infosys BPM scam — registration fee demanded     55                  14
   P007  ABC Career Consulting               ABC Career Consulting is a scam company     74                  23
   P008          WhatsApp Jobs      Jobs offered via WhatsApp groups are mos

---
## STEP 03 — Data Cleaning & Normalization

Normalize text fields to enable consistent matching:
- Convert to **lowercase**
- Remove **extra whitespace**
- Strip **special characters** (optional, preserved for readability)
- Standardize `company_name`, `title`, `content`, `comment`

In [19]:
# ============================================================
#  STEP 03 — DATA CLEANING & NORMALIZATION
# ============================================================

def normalize_text(text):
    """
    Normalize a text string for search indexing.
    
    Steps:
      1. Convert to string (handle NaN safely)
      2. Convert to lowercase
      3. Remove extra whitespace
      4. Strip leading/trailing spaces
    
    Returns: str — cleaned, lowercase, whitespace-normalized text
    """
    if pd.isna(text) or text is None:
        return ""
    text = str(text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)   # collapse multiple spaces/tabs/newlines
    text = text.strip()
    return text


# ----------------------------------------------------------
#  Clean community_posts
# ----------------------------------------------------------
print(" Cleaning community_posts...")
df_posts['company_name_clean'] = df_posts['company_name'].apply(normalize_text)
df_posts['title_clean']        = df_posts['title'].apply(normalize_text)
df_posts['content_clean']      = df_posts['content'].apply(normalize_text)
df_posts['tags_clean']         = df_posts['tags'].apply(normalize_text)
print(f"   Posts cleaned    — {len(df_posts)} rows")

# ----------------------------------------------------------
#  Clean community_comments
# ----------------------------------------------------------
print("🧹 Cleaning community_comments...")
df_comments['company_name_clean'] = df_comments['company_name'].apply(normalize_text)
df_comments['comment_clean']      = df_comments['comment'].apply(normalize_text)
print(f"  Comments cleaned — {len(df_comments)} rows")

# ----------------------------------------------------------
#  Clean scam_reports
# ----------------------------------------------------------
print("🧹 Cleaning scam_reports...")
df_reports['company_name_clean']  = df_reports['company_name'].apply(normalize_text)
df_reports['description_clean']   = df_reports['description'].apply(normalize_text)
df_reports['report_type_clean']   = df_reports['report_type'].apply(normalize_text)
print(f"   Reports cleaned  — {len(df_reports)} rows")

# ----------------------------------------------------------
#  Verification sample
# ----------------------------------------------------------
print("\n Normalization Verification Sample:")
print("-" * 60)
sample = df_posts[['company_name', 'company_name_clean']].head(4)
for _, row in sample.iterrows():
    print(f"  Original : {row['company_name']}")
    print(f"  Cleaned  : {row['company_name_clean']}")
    print()

print("Data cleaning complete for all three datasets.")

 Cleaning community_posts...
   Posts cleaned    — 12 rows
🧹 Cleaning community_comments...
  Comments cleaned — 14 rows
🧹 Cleaning scam_reports...
   Reports cleaned  — 8 rows

 Normalization Verification Sample:
------------------------------------------------------------
  Original : ABC Consulting
  Cleaned  : abc consulting

  Original : ABC Consultancy
  Cleaned  : abc consultancy

  Original : TCS
  Cleaned  : tcs

  Original : TCS
  Cleaned  : tcs

Data cleaning complete for all three datasets.


---
## STEP 04 — Build Search Corpus

Combine relevant text fields into a single `search_text` column per record.
This unified corpus enables single-pass keyword scanning.

```
search_text = company_name + " | " + title + " | " + content/comment + " | " + tags
```

In [20]:
# ============================================================
#  STEP 04 — BUILD SEARCH CORPUS
# ============================================================

# ----------------------------------------------------------
#  Posts corpus:
#  company_name | title | content | tags
# ----------------------------------------------------------
df_posts['search_text'] = (
    df_posts['company_name_clean'] + " | " +
    df_posts['title_clean']        + " | " +
    df_posts['content_clean']      + " | " +
    df_posts['tags_clean']
)

# ----------------------------------------------------------
#  Comments corpus:
#  company_name | comment
# ----------------------------------------------------------
df_comments['search_text'] = (
    df_comments['company_name_clean'] + " | " +
    df_comments['comment_clean']
)

# ----------------------------------------------------------
#  Reports corpus:
#  company_name | report_type | description
# ----------------------------------------------------------
df_reports['search_text'] = (
    df_reports['company_name_clean']  + " | " +
    df_reports['report_type_clean']   + " | " +
    df_reports['description_clean']
)

# ----------------------------------------------------------
#  Build unique company name sets for fuzzy matching
# ----------------------------------------------------------
all_company_names = (
    list(df_posts['company_name_clean'].unique()) +
    list(df_comments['company_name_clean'].unique()) +
    list(df_reports['company_name_clean'].unique())
)
# Deduplicate while preserving order
seen = set()
unique_company_names = []
for name in all_company_names:
    if name not in seen and name != "":
        seen.add(name)
        unique_company_names.append(name)

print(" Search Corpus Built Successfully")
print("-" * 60)
print(f"  Posts    corpus size : {len(df_posts)} records")
print(f"  Comments corpus size : {len(df_comments)} records")
print(f"  Reports  corpus size : {len(df_reports)} records")
print(f"  Unique company names : {len(unique_company_names)}")
print()
print(" Unique Company Names in Corpus:")
for i, name in enumerate(unique_company_names, 1):
    print(f"  {i:>2}. {name}")

print("\n Sample search_text (Posts):")
for i, txt in enumerate(df_posts['search_text'].head(3), 1):
    print(f"  [{i}] {txt[:100]}...")

 Search Corpus Built Successfully
------------------------------------------------------------
  Posts    corpus size : 12 records
  Comments corpus size : 14 records
  Reports  corpus size : 8 records
  Unique company names : 10

 Unique Company Names in Corpus:
   1. abc consulting
   2. abc consultancy
   3. tcs
   4. infosys
   5. infosys bpm
   6. abc career consulting
   7. whatsapp jobs
   8. global placement hub
   9. abc consulting pvt ltd
  10. tcs ion

 Sample search_text (Posts):
  [1] abc consulting | beware of abc consulting fake internship offer | they asked me to pay a registratio...
  [2] abc consultancy | abc consultancy demanded money before joining | abc consultancy pvt ltd sent me an...
  [3] tcs | fake tcs offer letter circulating on whatsapp | someone is sending tcs offer letters via whats...


---
## STEP 05 — Exact Match Search

A fast, case-insensitive keyword search across the `search_text` corpus.
Supports multiple space-separated tokens — all tokens must appear.

In [21]:
# ============================================================
#  STEP 05 — EXACT MATCH SEARCH
# ============================================================

def exact_search(query, df, source_type):
    """
    Search a DataFrame for records containing all keywords in the query.
    
    Parameters:
    -----------
    query       : str  — user search query (e.g., "ABC Consulting")
    df          : pd.DataFrame — the dataset to search
    source_type : str  — label: 'post', 'comment', 'report'
    
    Returns:
    --------
    pd.DataFrame — matching records with match_type='exact' and similarity_score=100
    """
    query_clean = normalize_text(query)
    tokens = query_clean.split()   # split into individual keywords
    
    if not tokens:
        return pd.DataFrame()
    
    # All tokens must appear in the search_text
    mask = df['search_text'].apply(
        lambda text: all(token in text for token in tokens)
    )
    
    results = df[mask].copy()
    results['match_type']       = 'exact'
    results['similarity_score'] = 100
    results['source_type']      = source_type
    
    return results


# ----------------------------------------------------------
#  Quick demo test
# ----------------------------------------------------------

print(" Exact Match Search — Demo")
print("=" * 50)

demo_query = "ABC Consulting"
print(f"Query: \"{demo_query}\"\n")

ex_posts    = exact_search(demo_query, df_posts,    'post')
ex_comments = exact_search(demo_query, df_comments, 'comment')
ex_reports  = exact_search(demo_query, df_reports,  'report')

print(f"  Posts    matched : {len(ex_posts)}")
print(f"  Comments matched : {len(ex_comments)}")
print(f"  Reports  matched : {len(ex_reports)}")

if len(ex_posts) > 0:
    print("\n  Matching Posts:")
    for _, row in ex_posts.iterrows():
        print(f"    [{row['post_id']}] {row['title']}")

if len(ex_comments) > 0:
    print("\n  Matching Comments:")
    for _, row in ex_comments.iterrows():
        print(f"    [{row['comment_id']}] {row['comment'][:70]}...")

if len(ex_reports) > 0:
    print("\n  Matching Reports:")
    for _, row in ex_reports.iterrows():
        print(f"    [{row['report_id']}] {row['company_name']} — {row['report_type']}")

print("\n Exact match search function ready.")

 Exact Match Search — Demo
Query: "ABC Consulting"

  Posts    matched : 3
  Comments matched : 5
  Reports  matched : 3

  Matching Posts:
    [P001] Beware of ABC Consulting fake internship offer
    [P007] ABC Career Consulting is a scam company
    [P010] ABC Consulting Pvt Ltd — another victim here

  Matching Comments:
    [C001] I also got the same message from ABC Consulting. Definitely a scam!...
    [C002] Report them to cybercrime.gov.in. ABC Consulting is listed there alrea...
    [C008] ABC Career Consulting is same group as ABC Consulting. Different names...
    [C010] They changed name to ABC Consulting Pvt Ltd but same phone number. Sca...
    [C013] Registration fee is the #1 red flag. Any company asking this is a scam...

  Matching Reports:
    [R001] ABC Consulting — Fake Internship
    [R005] ABC Career Consulting — Placement Scam
    [R007] ABC Consulting Pvt Ltd — Fake Internship

 Exact match search function ready.


---
## STEP 06 — Fuzzy Match Search

Uses `rapidfuzz` to find records that are *similar* to the query,
even when names are spelled differently.

**Strategy:**
- `fuzz.partial_ratio` — handles abbreviations and partial names
- `fuzz.token_set_ratio` — handles word-order differences
- Combined score with configurable **threshold (default: 70)**

In [22]:
# ============================================================
#  STEP 06 — FUZZY MATCH SEARCH
# ============================================================

FUZZY_THRESHOLD = 70   # Minimum similarity score (0–100) to include a result


def compute_fuzzy_score(query_clean, text):
    """
    Compute a combined fuzzy similarity score between query and a text field.
    
    Combines:
      - partial_ratio     : good for substrings / abbreviations
      - token_set_ratio   : good for word-order variations
    
    Returns:
    --------
    float — max of the two scorer values (0–100)
    """
    score_partial   = fuzz.partial_ratio(query_clean, text)
    score_token_set = fuzz.token_set_ratio(query_clean, text)
    return max(score_partial, score_token_set)


def fuzzy_search(query, df, source_type, threshold=FUZZY_THRESHOLD):
    """
    Fuzzy search across a DataFrame's search_text corpus.
    
    Parameters:
    -----------
    query       : str           — user search query
    df          : pd.DataFrame  — dataset to search
    source_type : str           — 'post', 'comment', or 'report'
    threshold   : int           — minimum fuzzy score (default 70)
    
    Returns:
    --------
    pd.DataFrame — matching records with similarity_score and match_type='fuzzy'
    """
    query_clean = normalize_text(query)
    
    if not query_clean:
        return pd.DataFrame()
    
    # Compute similarity score for every record in the corpus
    scores = df['search_text'].apply(
        lambda text: compute_fuzzy_score(query_clean, text)
    )
    
    # Filter by threshold
    mask    = scores >= threshold
    results = df[mask].copy()
    results['similarity_score'] = scores[mask].values
    results['match_type']       = 'fuzzy'
    results['source_type']      = source_type
    
    # Sort by similarity descending
    results = results.sort_values('similarity_score', ascending=False)
    
    return results


# ----------------------------------------------------------
#  Demo: fuzzy variations of ABC Consulting
# ----------------------------------------------------------

print("Fuzzy Match Search — Demo")
print("=" * 60)
print(f"Threshold : {FUZZY_THRESHOLD}+\n")

fuzzy_queries = ["ABC Consulting", "ABC Consultancy", "ABC Career", "tcs", "infosys"]

for qry in fuzzy_queries:
    hits = fuzzy_search(qry, df_posts, 'post')
    print(f"  Query: \"{qry}\"  →  {len(hits)} post(s) matched")
    for _, row in hits.head(2).iterrows():
        print(f"    [{row['post_id']}] Score={row['similarity_score']:>3}  {row['title'][:55]}...")
    print()

print(" Fuzzy search function ready.")

Fuzzy Match Search — Demo
Threshold : 70+

  Query: "ABC Consulting"  →  4 post(s) matched
    [P001] Score=100.0  Beware of ABC Consulting fake internship offer...
    [P007] Score=100.0  ABC Career Consulting is a scam company...

  Query: "ABC Consultancy"  →  3 post(s) matched
    [P002] Score=100.0  ABC Consultancy demanded money before joining...
    [P001] Score=85.71428571428572  Beware of ABC Consulting fake internship offer...

  Query: "ABC Career"  →  1 post(s) matched
    [P007] Score=100.0  ABC Career Consulting is a scam company...

  Query: "tcs"  →  3 post(s) matched
    [P003] Score=100.0  Fake TCS offer letter circulating on WhatsApp...
    [P004] Score=100.0  TCS does not charge any fees — official clarification...

  Query: "infosys"  →  3 post(s) matched
    [P005] Score=100.0  Fake Infosys internship on LinkedIn...
    [P006] Score=100.0  Infosys BPM scam — registration fee demanded...

 Fuzzy search function ready.


---
## STEP 07 — Company Search Engine

The central search function that combines **exact** and **fuzzy** matching
across **all three data sources** (posts, comments, reports).

In [23]:
# ============================================================
#  STEP 07 — COMPANY SEARCH ENGINE
# ============================================================

def search_company(company_name, threshold=FUZZY_THRESHOLD, verbose=False):
    """
    Master search function — searches posts, comments, and reports
    using both exact and fuzzy matching.
    
    Strategy:
      1. Exact match first (score = 100)
      2. Fuzzy match for missed/variant names (score >= threshold)
      3. Deduplicate by record ID
    
    Parameters:
    -----------
    company_name : str  — search query from user
    threshold    : int  — fuzzy match minimum score (default 70)
    verbose      : bool — print debug info if True
    
    Returns:
    --------
    dict with keys: posts, comments, reports
         each containing a ranked DataFrame of matching records
    """
    
    def merge_results(exact_df, fuzzy_df, id_col):
        """
        Merge exact and fuzzy results, keeping exact match record
        when a duplicate is found (exact always wins).
        """
        if exact_df.empty and fuzzy_df.empty:
            return pd.DataFrame()
        
        if exact_df.empty:
            return fuzzy_df
        
        if fuzzy_df.empty:
            return exact_df
        
        # Stack and deduplicate — keep first (exact) on duplicate IDs
        combined = pd.concat([exact_df, fuzzy_df], ignore_index=True)
        combined = combined.drop_duplicates(subset=[id_col], keep='first')
        return combined
    
    if verbose:
        print(f"\n Searching for: \"{company_name}\"")
        print(f"   Threshold : {threshold}")
    
    # ---------- POSTS ----------
    exact_posts = exact_search(company_name, df_posts, 'post')
    fuzzy_posts = fuzzy_search(company_name, df_posts, 'post', threshold)
    result_posts = merge_results(exact_posts, fuzzy_posts, 'post_id')
    
    # ---------- COMMENTS ----------
    exact_comments = exact_search(company_name, df_comments, 'comment')
    fuzzy_comments = fuzzy_search(company_name, df_comments, 'comment', threshold)
    result_comments = merge_results(exact_comments, fuzzy_comments, 'comment_id')
    
    # ---------- REPORTS ----------
    exact_reports = exact_search(company_name, df_reports, 'report')
    fuzzy_reports = fuzzy_search(company_name, df_reports, 'report', threshold)
    result_reports = merge_results(exact_reports, fuzzy_reports, 'report_id')
    
    if verbose:
        print(f"   Posts    found : {len(result_posts)}")
        print(f"   Comments found : {len(result_comments)}")
        print(f"   Reports  found : {len(result_reports)}")
    
    return {
        'posts':    result_posts,
        'comments': result_comments,
        'reports':  result_reports
    }


# ----------------------------------------------------------
#  Demo
# ----------------------------------------------------------

print(" Company Search Engine — Demo")
print("=" * 60)

demo_results = search_company("ABC Consulting", verbose=True)

print("\n  Matched Posts:")
if not demo_results['posts'].empty:
    print(demo_results['posts'][['post_id','company_name','title','match_type','similarity_score']].to_string(index=False))

print("\n  Matched Comments:")
if not demo_results['comments'].empty:
    print(demo_results['comments'][['comment_id','company_name','comment','match_type','similarity_score']].head(4).to_string(index=False))

print("\n  Matched Reports:")
if not demo_results['reports'].empty:
    print(demo_results['reports'][['report_id','company_name','report_type','severity','match_type','similarity_score']].to_string(index=False))

print("\n Company search engine ready.")

 Company Search Engine — Demo

 Searching for: "ABC Consulting"
   Threshold : 70
   Posts    found : 4
   Comments found : 6
   Reports  found : 4

  Matched Posts:
post_id           company_name                                          title match_type  similarity_score
   P001         ABC Consulting Beware of ABC Consulting fake internship offer      exact        100.000000
   P007  ABC Career Consulting        ABC Career Consulting is a scam company      exact        100.000000
   P010 ABC Consulting Pvt Ltd   ABC Consulting Pvt Ltd — another victim here      exact        100.000000
   P002        ABC Consultancy  ABC Consultancy demanded money before joining      fuzzy         88.888889

  Matched Comments:
comment_id           company_name                                                                            comment match_type  similarity_score
      C001         ABC Consulting                I also got the same message from ABC Consulting. Definitely a scam!      exact     

---
## STEP 08 — Popularity Ranking

Rank results so that **most-discussed** and **most-reported** companies appear first.

**Ranking formula:**
```
popularity_score = (likes × 0.3) + (comments × 0.4) + (upvotes × 0.3)
reports_score    = number_of_reports (weighted highest)
```

In [24]:
# ============================================================
#  STEP 08 — POPULARITY RANKING
# ============================================================

def rank_posts(df):
    """
    Rank post results by popularity.
    
    Score = 0.3 * likes + 0.4 * number_of_comments + 0.3 * similarity_score
    """
    if df.empty:
        return df
    
    df = df.copy()
    df['likes']               = pd.to_numeric(df.get('likes', 0), errors='coerce').fillna(0)
    df['number_of_comments']  = pd.to_numeric(df.get('number_of_comments', 0), errors='coerce').fillna(0)
    df['similarity_score']    = pd.to_numeric(df.get('similarity_score', 0), errors='coerce').fillna(0)
    
    df['popularity_score'] = (
        df['likes']              * 0.30 +
        df['number_of_comments'] * 0.40 +
        df['similarity_score']   * 0.30
    ).round(2)
    
    return df.sort_values(['popularity_score', 'similarity_score'], ascending=False)


def rank_comments(df):
    """
    Rank comment results by upvotes and similarity.
    
    Score = 0.6 * upvotes + 0.4 * similarity_score
    """
    if df.empty:
        return df
    
    df = df.copy()
    df['upvotes']          = pd.to_numeric(df.get('upvotes', 0), errors='coerce').fillna(0)
    df['similarity_score'] = pd.to_numeric(df.get('similarity_score', 0), errors='coerce').fillna(0)
    
    df['popularity_score'] = (
        df['upvotes']          * 0.60 +
        df['similarity_score'] * 0.40
    ).round(2)
    
    return df.sort_values(['popularity_score', 'similarity_score'], ascending=False)


def rank_reports(df):
    """
    Rank report results by number_of_reports and severity.
    
    Severity weights: Critical=3, High=2, Medium=1, Low=0.5
    Score = number_of_reports * severity_weight
    """
    if df.empty:
        return df
    
    severity_weights = {'critical': 3, 'high': 2, 'medium': 1, 'low': 0.5}
    
    df = df.copy()
    df['number_of_reports'] = pd.to_numeric(df.get('number_of_reports', 0), errors='coerce').fillna(0)
    df['severity_weight']   = df['severity'].str.lower().map(severity_weights).fillna(1)
    
    df['popularity_score'] = (
        df['number_of_reports'] * df['severity_weight']
    ).round(2)
    
    return df.sort_values(['popularity_score', 'number_of_reports'], ascending=False)


# ----------------------------------------------------------
#  Apply ranking to demo results
# ----------------------------------------------------------

print("📊 Popularity Ranking — Demo")
print("=" * 60)

ranked_posts    = rank_posts(demo_results['posts'])
ranked_comments = rank_comments(demo_results['comments'])
ranked_reports  = rank_reports(demo_results['reports'])

print("\n🏆 Ranked Posts (by popularity_score):")
if not ranked_posts.empty:
    print(ranked_posts[['post_id','company_name','title','likes','number_of_comments','similarity_score','popularity_score']].to_string(index=False))

print("\n🏆 Ranked Reports (by popularity_score):")
if not ranked_reports.empty:
    print(ranked_reports[['report_id','company_name','report_type','severity','number_of_reports','popularity_score']].to_string(index=False))

print("\n Popularity ranking functions ready.")

📊 Popularity Ranking — Demo

🏆 Ranked Posts (by popularity_score):
post_id           company_name                                          title  likes  number_of_comments  similarity_score  popularity_score
   P007  ABC Career Consulting        ABC Career Consulting is a scam company     74                  23        100.000000             61.40
   P001         ABC Consulting Beware of ABC Consulting fake internship offer     45                  12        100.000000             48.30
   P010 ABC Consulting Pvt Ltd   ABC Consulting Pvt Ltd — another victim here     41                  11        100.000000             46.70
   P002        ABC Consultancy  ABC Consultancy demanded money before joining     38                   9         88.888889             41.67

🏆 Ranked Reports (by popularity_score):
report_id           company_name      report_type severity  number_of_reports  popularity_score
     R005  ABC Career Consulting   Placement Scam     High                 11              

---
## STEP 09 — Community Intelligence Summary

Generate aggregate statistics and an **overall sentiment score** for a query.

- `total_posts` — how many posts mention this company
- `total_comments` — total engagement in comments
- `total_reports` — number of scam reports filed
- `overall_sentiment_score` — derived from report count, severity, and upvotes

In [25]:
# ============================================================
#  STEP 09 — COMMUNITY INTELLIGENCE SUMMARY
# ============================================================

def compute_sentiment_score(posts_df, comments_df, reports_df):
    """
    Compute an overall community sentiment score (0–10).
    
    Logic:
      - Starts at 5.0 (neutral)
      - Penalised for each verified scam report (severity-weighted)
      - Penalised for high number_of_reports
      - Boosted slightly by verified = False (unconfirmed reports)
    
    Scale:
      0.0 – 3.0  : Highly dangerous / confirmed scam
      3.0 – 5.0  : Suspicious / multiple reports
      5.0 – 7.0  : Mixed / low reports
      7.0 – 10.0 : Likely safe (few or no reports)
    """
    score = 5.0
    
    if not reports_df.empty:
        severity_weights = {'critical': 2.0, 'high': 1.2, 'medium': 0.6, 'low': 0.3}
        
        for _, row in reports_df.iterrows():
            w  = severity_weights.get(str(row.get('severity', '')).lower(), 0.5)
            nr = float(row.get('number_of_reports', 1))
            verified = bool(row.get('verified', False))
            
            if verified:
                score -= w * (1 + np.log1p(nr) * 0.3)
            else:
                score -= w * 0.3
    
    # Clip to valid range [0, 10]
    score = round(max(0.0, min(10.0, score)), 2)
    return score


def community_intelligence_summary(query, posts_df, comments_df, reports_df):
    """
    Generate a full community intelligence summary for a search query.
    
    Parameters:
    -----------
    query       : str  — search term
    posts_df    : pd.DataFrame — matched posts
    comments_df : pd.DataFrame — matched comments
    reports_df  : pd.DataFrame — matched reports
    
    Returns:
    --------
    dict — summary statistics
    """
    total_posts    = len(posts_df)
    total_comments = len(comments_df)
    total_reports  = len(reports_df)
    
    total_likes   = int(pd.to_numeric(posts_df.get('likes', pd.Series()), errors='coerce').sum())
    total_upvotes = int(pd.to_numeric(comments_df.get('upvotes', pd.Series()), errors='coerce').sum())
    total_report_count = int(pd.to_numeric(reports_df.get('number_of_reports', pd.Series()), errors='coerce').sum())
    
    # Severity breakdown
    severity_counts = {}
    if not reports_df.empty and 'severity' in reports_df.columns:
        severity_counts = reports_df['severity'].value_counts().to_dict()
    
    verified_count = int(reports_df['verified'].sum()) if (not reports_df.empty and 'verified' in reports_df.columns) else 0
    
    # Sentiment score
    sentiment_score = compute_sentiment_score(posts_df, comments_df, reports_df)
    
    # Danger level label
    if sentiment_score <= 2.0:
        danger_level = "CONFIRMED SCAM"
    elif sentiment_score <= 4.0:
        danger_level = "HIGH RISK"
    elif sentiment_score <= 6.0:
        danger_level = "SUSPICIOUS"
    elif sentiment_score <= 8.0:
        danger_level = "LOW RISK"
    else:
        danger_level = "APPEARS SAFE"
    
    return {
        "query":                  query,
        "total_posts":            total_posts,
        "total_comments":         total_comments,
        "total_reports":          total_reports,
        "total_likes":            total_likes,
        "total_upvotes":          total_upvotes,
        "total_report_count":     total_report_count,
        "severity_breakdown":     severity_counts,
        "verified_reports":       verified_count,
        "overall_sentiment_score":sentiment_score,
        "danger_level":           danger_level,
        "generated_at":           datetime.now().isoformat()
    }


# ----------------------------------------------------------
#  Demo
# ----------------------------------------------------------

print(" Community Intelligence Summary — Demo")
print("=" * 60)

summary = community_intelligence_summary(
    "ABC Consulting",
    ranked_posts,
    ranked_comments,
    ranked_reports
)

for k, v in summary.items():
    print(f"  {k:<30} : {v}")

print("\n Community intelligence summary ready.")

 Community Intelligence Summary — Demo
  query                          : ABC Consulting
  total_posts                    : 4
  total_comments                 : 6
  total_reports                  : 4
  total_likes                    : 198
  total_upvotes                  : 163
  total_report_count             : 30
  severity_breakdown             : {'High': 4}
  verified_reports               : 3
  overall_sentiment_score        : 0.0
  danger_level                   : CONFIRMED SCAM
  generated_at                   : 2026-06-13T00:21:50.956380

 Community intelligence summary ready.


---
## STEP 10 — API-Style Response

Bundle the search results and summary into a **structured JSON response** —
ready for use by the Community Page frontend or API consumers.

In [26]:
# ============================================================
#  STEP 10 — API-STYLE RESPONSE
# ============================================================

def build_api_response(query, threshold=FUZZY_THRESHOLD):
    """
    Full pipeline — search → rank → summarize → return API response.
    
    Parameters:
    -----------
    query     : str  — company name or search keyword
    threshold : int  — fuzzy match threshold (default 70)
    
    Returns:
    --------
    dict — API-style JSON response with:
           query, summary stats, ranked results (posts, comments, reports)
    """
    
    # 1. Search
    raw = search_company(query, threshold)
    
    # 2. Rank
    posts    = rank_posts(raw['posts'])
    comments = rank_comments(raw['comments'])
    reports  = rank_reports(raw['reports'])
    
    # 3. Summarize
    summary = community_intelligence_summary(query, posts, comments, reports)
    
    # 4. Serialize results to list-of-dicts
    def df_to_records(df, cols):
        """Convert a DataFrame subset to a list of dicts for JSON output."""
        if df.empty:
            return []
        available_cols = [c for c in cols if c in df.columns]
        return df[available_cols].fillna("").to_dict(orient='records')
    
    post_cols    = ['post_id','company_name','title','content','likes',
                    'number_of_comments','date','match_type','similarity_score','popularity_score']
    comment_cols = ['comment_id','post_id','company_name','comment','upvotes',
                    'date','match_type','similarity_score','popularity_score']
    report_cols  = ['report_id','company_name','report_type','description','severity',
                    'number_of_reports','verified','date','match_type','similarity_score','popularity_score']
    
    response = {
        "query":                   query,
        "total_posts":             summary['total_posts'],
        "total_comments":          summary['total_comments'],
        "total_reports":           summary['total_reports'],
        "total_likes":             summary['total_likes'],
        "total_upvotes":           summary['total_upvotes'],
        "total_report_count":      summary['total_report_count'],
        "verified_reports":        summary['verified_reports'],
        "severity_breakdown":      summary['severity_breakdown'],
        "overall_sentiment_score": summary['overall_sentiment_score'],
        "danger_level":            summary['danger_level'],
        "generated_at":            summary['generated_at'],
        "results": {
            "posts":    df_to_records(posts,    post_cols),
            "comments": df_to_records(comments, comment_cols),
            "reports":  df_to_records(reports,  report_cols)
        }
    }
    
    return response


# ----------------------------------------------------------
#  Demo API response
# ----------------------------------------------------------

print("📡 API-Style Response — Demo")
print("=" * 60)

api_response = build_api_response("ABC Consulting")

# Print compact summary (not full results)
summary_keys = [k for k in api_response if k != 'results']
compact = {k: api_response[k] for k in summary_keys}
print(json.dumps(compact, indent=2))

print(f"\n  results.posts    → {len(api_response['results']['posts'])} records")
print(f"  results.comments → {len(api_response['results']['comments'])} records")
print(f"  results.reports  → {len(api_response['results']['reports'])} records")
print("\n API response builder ready.")

📡 API-Style Response — Demo
{
  "query": "ABC Consulting",
  "total_posts": 4,
  "total_comments": 6,
  "total_reports": 4,
  "total_likes": 198,
  "total_upvotes": 163,
  "total_report_count": 30,
  "verified_reports": 3,
  "severity_breakdown": {
    "High": 4
  },
  "overall_sentiment_score": 0.0,
  "danger_level": "CONFIRMED SCAM",
  "generated_at": "2026-06-13T00:21:51.108787"
}

  results.posts    → 4 records
  results.comments → 6 records
  results.reports  → 4 records

 API response builder ready.


---
## STEP 11 — Test Cases

Test the complete search pipeline with representative queries:
1. `ABC Consulting`
2. `TCS`
3. `Infosys`
4. `Registration Fee`
5. `WhatsApp`

In [27]:
# ============================================================
#  STEP 11 — TEST CASES
# ============================================================

TEST_QUERIES = [
    "ABC Consulting",
    "TCS",
    "Infosys",
    "Registration Fee",
    "WhatsApp"
]

SEPARATOR = "=" * 70
all_responses = {}

for i, query in enumerate(TEST_QUERIES, 1):
    print(SEPARATOR)
    print(f"  TEST CASE {i}/{len(TEST_QUERIES)} : \"{query}\"")
    print(SEPARATOR)
    
    response = build_api_response(query)
    all_responses[query] = response
    
    # ── Summary Stats ──
    print(f"  {'Query':<26}: {response['query']}")
    print(f"  {'Total Posts':<26}: {response['total_posts']}")
    print(f"  {'Total Comments':<26}: {response['total_comments']}")
    print(f"  {'Total Reports':<26}: {response['total_reports']}")
    print(f"  {'Total Likes':<26}: {response['total_likes']}")
    print(f"  {'Total Upvotes':<26}: {response['total_upvotes']}")
    print(f"  {'Report Count (aggregate)':<26}: {response['total_report_count']}")
    print(f"  {'Verified Reports':<26}: {response['verified_reports']}")
    print(f"  {'Severity Breakdown':<26}: {response['severity_breakdown']}")
    print(f"  {'Sentiment Score':<26}: {response['overall_sentiment_score']} / 10.0")
    print(f"  {'Danger Level':<26}: {response['danger_level']}")
    
    # ── Top Matching Posts ──
    if response['results']['posts']:
        print(f"\n   Top Posts:")
        for p in response['results']['posts'][:3]:
            score_display = f"{p.get('popularity_score','N/A'):>6}"
            print(f"    [{p.get('post_id','?')}] Score={score_display}  {str(p.get('title',''))[:55]}")
    
    # ── Top Matching Reports ──
    if response['results']['reports']:
        print(f"\n   Top Reports:")
        for r in response['results']['reports'][:3]:
            print(f"    [{r.get('report_id','?')}] {r.get('severity','?'):<10} {r.get('company_name','')} — {r.get('report_type','')}")
    
    # ── Top Matching Comments ──
    if response['results']['comments']:
        print(f"\n   Top Comments:")
        for c in response['results']['comments'][:2]:
            print(f"    [{c.get('comment_id','?')}] {str(c.get('comment',''))[:65]}")
    
    print()

print(SEPARATOR)
print("All test cases completed.")
print(SEPARATOR)

  TEST CASE 1/5 : "ABC Consulting"
  Query                     : ABC Consulting
  Total Posts               : 4
  Total Comments            : 6
  Total Reports             : 4
  Total Likes               : 198
  Total Upvotes             : 163
  Report Count (aggregate)  : 30
  Verified Reports          : 3
  Severity Breakdown        : {'High': 4}
  Sentiment Score           : 0.0 / 10.0
  Danger Level              : CONFIRMED SCAM

   Top Posts:
    [P007] Score=  61.4  ABC Career Consulting is a scam company
    [P001] Score=  48.3  Beware of ABC Consulting fake internship offer
    [P010] Score=  46.7  ABC Consulting Pvt Ltd — another victim here

   Top Reports:
    [R005] High       ABC Career Consulting — Placement Scam
    [R001] High       ABC Consulting — Fake Internship
    [R007] High       ABC Consulting Pvt Ltd — Fake Internship

   Top Comments:
    [C013] Registration fee is the #1 red flag. Any company asking this is a
    [C008] ABC Career Consulting is same group as 

In [28]:
# ----------------------------------------------------------
#  Test Summary Table
# ----------------------------------------------------------

print("\n Test Summary Table")
print("-" * 90)
print(f"{'Query':<25} {'Posts':>6} {'Comments':>9} {'Reports':>8} {'Sentiment':>10} {'Danger Level':<22}")
print("-" * 90)

for query, resp in all_responses.items():
    print(
        f"{query:<25} "
        f"{resp['total_posts']:>6} "
        f"{resp['total_comments']:>9} "
        f"{resp['total_reports']:>8} "
        f"{resp['overall_sentiment_score']:>10.2f} "
        f"{resp['danger_level']:<22}"
    )

print("-" * 90)
print("\n Test case summary complete.")


 Test Summary Table
------------------------------------------------------------------------------------------
Query                      Posts  Comments  Reports  Sentiment Danger Level          
------------------------------------------------------------------------------------------
ABC Consulting                 4         6        4       0.00 CONFIRMED SCAM        
TCS                            3         3        2       0.00 CONFIRMED SCAM        
Infosys                        3         3        1       1.20 CONFIRMED SCAM        
Registration Fee               4         1        2       0.00 CONFIRMED SCAM        
WhatsApp                       2         3        2       0.00 CONFIRMED SCAM        
------------------------------------------------------------------------------------------

 Test case summary complete.


---
## STEP 12 — Export Search Index

Save the unified search corpus to `../data/community_search_index.csv`
for use by the frontend Community Search Bar.

In [29]:
# ============================================================
#  STEP 12 — EXPORT SEARCH INDEX
# ============================================================

OUTPUT_PATH = os.path.join(DATA_DIR, "community_search_index.csv")

# ----------------------------------------------------------
#  Build unified index with source type labels
# ----------------------------------------------------------

# Posts index
index_posts = df_posts[['post_id', 'company_name', 'title', 'content', 'date',
                         'likes', 'number_of_comments', 'tags',
                         'company_name_clean', 'search_text']].copy()
index_posts['source_type'] = 'post'
index_posts['record_id']   = index_posts['post_id']
index_posts['main_text']   = index_posts['title'] + " " + index_posts['content']
index_posts['engagement']  = index_posts['likes'] + index_posts['number_of_comments']

# Comments index
index_comments = df_comments[['comment_id', 'post_id', 'company_name', 'comment',
                               'date', 'upvotes',
                               'company_name_clean', 'search_text']].copy()
index_comments['source_type'] = 'comment'
index_comments['record_id']   = index_comments['comment_id']
index_comments['main_text']   = index_comments['comment']
index_comments['engagement']  = index_comments['upvotes']
index_comments['title']       = "[Comment on Post: " + index_comments['post_id'] + "]"
index_comments['likes']       = 0
index_comments['number_of_comments'] = 0
index_comments['tags']        = ""

# Reports index
index_reports = df_reports[['report_id', 'company_name', 'report_type', 'description',
                             'date', 'severity', 'number_of_reports', 'verified',
                             'company_name_clean', 'search_text']].copy()
index_reports['source_type'] = 'report'
index_reports['record_id']   = index_reports['report_id']
index_reports['main_text']   = index_reports['report_type'] + " — " + index_reports['description']
index_reports['engagement']  = index_reports['number_of_reports']
index_reports['title']       = "[Report] " + index_reports['report_type']
index_reports['likes']       = 0
index_reports['number_of_comments'] = 0
index_reports['tags']        = index_reports['severity']
index_reports['upvotes']     = 0
index_reports['post_id']     = ""

# ----------------------------------------------------------
#  Unified schema
# ----------------------------------------------------------
UNIFIED_COLS = [
    'record_id', 'source_type', 'company_name', 'company_name_clean',
    'title', 'main_text', 'date', 'engagement', 'search_text',
    'tags'
]

# Align columns
for df_idx in [index_posts, index_comments, index_reports]:
    for col in UNIFIED_COLS:
        if col not in df_idx.columns:
            df_idx[col] = ""

search_index = pd.concat(
    [index_posts[UNIFIED_COLS],
     index_comments[UNIFIED_COLS],
     index_reports[UNIFIED_COLS]],
    ignore_index=True
)

# Sort by engagement descending
search_index['engagement'] = pd.to_numeric(search_index['engagement'], errors='coerce').fillna(0)
search_index = search_index.sort_values('engagement', ascending=False).reset_index(drop=True)

# ----------------------------------------------------------
#  Save
# ----------------------------------------------------------
search_index.to_csv(OUTPUT_PATH, index=False)

print(" Search Index Export")
print("=" * 60)
print(f"  Output path   : {OUTPUT_PATH}")
print(f"  Total records : {len(search_index)}")
print(f"  Columns       : {list(search_index.columns)}")
print()
print(" Source Type Breakdown:")
print(search_index['source_type'].value_counts().to_string())
print()
print(" Preview (top 5 by engagement):")
print(search_index[['record_id','source_type','company_name','title','engagement']].head(5).to_string(index=False))
print()
print(" Community Search Index exported successfully.")

 Search Index Export
  Output path   : ../data/community_search_index.csv
  Total records : 34
  Columns       : ['record_id', 'source_type', 'company_name', 'company_name_clean', 'title', 'main_text', 'date', 'engagement', 'search_text', 'tags']

 Source Type Breakdown:
source_type
comment    14
post       12
report      8

 Preview (top 5 by engagement):
record_id source_type  company_name                                                 title  engagement
     P008        post WhatsApp Jobs      Jobs offered via WhatsApp groups are mostly fake         165
     P003        post           TCS         Fake TCS offer letter circulating on WhatsApp         136
     P012        post       Infosys            How to verify genuine Infosys offer letter         126
     P011        post       TCS iON               Fake TCS iON internship offer via email         118
     P004        post           TCS TCS does not charge any fees — official clarification         108

 Community Search Index expo

---

## 📋 Notebook Summary

| Step | Description | Status |
|------|-------------|--------|
| 01 | Import Libraries | ✅ |
| 02 | Load Community Data (posts / comments / reports) | ✅ |
| 03 | Data Cleaning & Normalization | ✅ |
| 04 | Build Search Corpus (`search_text`) | ✅ |
| 05 | Exact Match Search (`exact_search()`) | ✅ |
| 06 | Fuzzy Match Search (`fuzzy_search()`, threshold 70+) | ✅ |
| 07 | Company Search Engine (`search_company()`) | ✅ |
| 08 | Popularity Ranking (likes / upvotes / reports) | ✅ |
| 09 | Community Intelligence Summary (sentiment score) | ✅ |
| 10 | API-Style JSON Response (`build_api_response()`) | ✅ |
| 11 | Test Cases (5 queries) | ✅ |
| 12 | Export `community_search_index.csv` | ✅ |

---

### 🔗 Integration Points

| Component | Usage |
|-----------|-------|
| Community Page Search Bar | Calls `build_api_response(query)` |
| Notebook 5 (Detection Engine) | Feeds `scam_reports.csv` |
| Notebook 6 (URL Analysis) | Enriches report descriptions |
| Frontend API | Consumes the JSON response dict directly |

---

### 📦 Exported Files

```
../data/community_posts.csv          ← source data
../data/community_comments.csv       ← source data  
../data/scam_reports.csv             ← source data
../data/community_search_index.csv   ← unified search index (this notebook)
```

---

*Notebook 7 — Community Search Engine | Fake Internship & Job Scam Detection System*